In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import numpy as np

df_train = pd.read_csv("/content/drive/My Drive/walmart_competition_data/train.csv")
df_test = pd.read_csv("/content/drive/My Drive/walmart_competition_data/test.csv")
df_features = pd.read_csv("/content/drive/My Drive/walmart_competition_data/features.csv")
df_stores = pd.read_csv("/content/drive/My Drive/walmart_competition_data/stores.csv")

df_train_merged = df_train.merge(df_stores, on='Store', how='left')
df_train_merged = df_train_merged.merge(df_features, on=['Store', 'Date', 'IsHoliday'], how='left')

train_df_split, val_df_split = train_test_split(
    df_train_merged, test_size=0.2, random_state=42
)

y_train = train_df_split['Weekly_Sales']
is_holiday_train = train_df_split['IsHoliday']
X_train = train_df_split.copy()

y_val = val_df_split['Weekly_Sales']
is_holiday_val = val_df_split['IsHoliday']
X_val = val_df_split.copy()



In [ ]:
X_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
138466,15,3,2011-04-01,7490.24,False,B,123737,30.34,3.811,NaN,NaN,NaN,NaN,NaN,134.068258,7.658
289214,30,25,2010-06-11,48.25,False,C,42988,83.51,2.668,NaN,NaN,NaN,NaN,NaN,211.112002,8.200
52351,6,27,2010-06-04,1262.75,False,A,202505,79.44,2.705,NaN,NaN,NaN,NaN,NaN,212.698244,7.092
203504,21,49,2011-12-02,8722.34,False,B,140167,48.72,3.172,3389.10,43.0,1258.32,325.35,8623.67,218.359032,7.441
233606,24,55,2012-01-06,15247.36,False,A,203819,32.86,3.585,7325.68,25367.9,203.51,1745.20,3261.35,136.698129,8.659


In [ ]:
from __future__ import annotations

from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


MARKDOWN_COLS = ("MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5")
NUMERIC_EXTERNAL_COLS = ("CPI", "Unemployment", "Temperature", "Fuel_Price")


def _existing_columns(frame: pd.DataFrame, columns: Iterable[str]) -> list[str]:
    return [col for col in columns if col in frame.columns]


class WalmartFeatureCleaner(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        date_col: str = "Date",
        markdown_cols: tuple[str, ...] = MARKDOWN_COLS,
        numeric_impute_cols: tuple[str, ...] = NUMERIC_EXTERNAL_COLS,
        add_markdown_missing_indicators: bool = True,
        markdown_fill_value: float = 0.0,
        numeric_impute_strategy: str = "median",
        category_cols: tuple[str, ...] = ("Store", "Dept", "Type"),
    ):
        self.date_col = date_col
        self.markdown_cols = markdown_cols
        self.numeric_impute_cols = numeric_impute_cols
        self.add_markdown_missing_indicators = add_markdown_missing_indicators
        self.markdown_fill_value = markdown_fill_value
        self.numeric_impute_strategy = numeric_impute_strategy
        self.category_cols = category_cols

    def fit(self, X: pd.DataFrame, y=None):
        if self.numeric_impute_strategy not in {"median", "mean", "none"}:
            raise ValueError("numeric_impute_strategy must be 'median', 'mean', or 'none'.")

        self.numeric_fill_values_ = {}
        numeric_cols = _existing_columns(X, self.numeric_impute_cols)
        if self.numeric_impute_strategy != "none":
            for col in numeric_cols:
                if self.numeric_impute_strategy == "median":
                    self.numeric_fill_values_[col] = X[col].median()
                else:
                    self.numeric_fill_values_[col] = X[col].mean()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()

        if self.date_col in frame.columns:
            frame[self.date_col] = pd.to_datetime(frame[self.date_col])

        for col in _existing_columns(frame, self.markdown_cols):
            if self.add_markdown_missing_indicators:
                frame[f"{col}_missing"] = frame[col].isna().astype("int8")
            frame[col] = frame[col].fillna(self.markdown_fill_value)

        for col, value in getattr(self, "numeric_fill_values_", {}).items():
            if col in frame.columns:
                frame[col] = frame[col].fillna(value)

        for col in _existing_columns(frame, self.category_cols):
            frame[col] = frame[col].astype("category")

        if "IsHoliday" in frame.columns:
            frame["IsHoliday"] = frame["IsHoliday"].astype("int8")

        return frame


class CalendarFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        date_col: str = "Date",
        start_date: str = "2010-02-05",
        add_cyclical_features: bool = True,
        drop_date: bool = False,
    ):
        self.date_col = date_col
        self.start_date = start_date
        self.add_cyclical_features = add_cyclical_features
        self.drop_date = drop_date

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        date = pd.to_datetime(frame[self.date_col])
        iso = date.dt.isocalendar()

        frame["Year"] = date.dt.year.astype("int16")
        frame["Month"] = date.dt.month.astype("int8")
        frame["WeekOfYear"] = iso.week.astype("int8")
        frame["Quarter"] = date.dt.quarter.astype("int8")
        frame["DayOfYear"] = date.dt.dayofyear.astype("int16")
        frame["DaysFromStart"] = (date - pd.Timestamp(self.start_date)).dt.days.astype("int16")

        if self.add_cyclical_features:
            frame["WeekSin"] = np.sin(2 * np.pi * frame["WeekOfYear"] / 52.0)
            frame["WeekCos"] = np.cos(2 * np.pi * frame["WeekOfYear"] / 52.0)
            frame["MonthSin"] = np.sin(2 * np.pi * frame["Month"] / 12.0)
            frame["MonthCos"] = np.cos(2 * np.pi * frame["Month"] / 12.0)

        if self.drop_date:
            frame = frame.drop(columns=[self.date_col])

        return frame


class WalmartHolidayFeatureTransformer(BaseEstimator, TransformerMixin):

    HOLIDAY_DATES = {
        "SuperBowl": ("2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"),
        "LaborDay": ("2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"),
        "Thanksgiving": ("2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"),
        "Christmas": ("2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"),
    }

    def __init__(
        self,
        date_col: str = "Date",
        add_holiday_flags: bool = True,
        add_proximity_features: bool = True,
    ):
        self.date_col = date_col
        self.add_holiday_flags = add_holiday_flags
        self.add_proximity_features = add_proximity_features

    def fit(self, X: pd.DataFrame, y=None):
        self.holiday_dates_ = {
            name: pd.to_datetime(list(dates)) for name, dates in self.HOLIDAY_DATES.items()
        }
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        date = pd.to_datetime(frame[self.date_col])

        for name, holiday_dates in self.holiday_dates_.items():
            if self.add_holiday_flags:
                frame[f"Is{name}Week"] = date.isin(holiday_dates).astype("int8")

            if self.add_proximity_features:
                distances = np.vstack([(date - holiday).dt.days.to_numpy() for holiday in holiday_dates])
                nearest_distance = distances[np.abs(distances).argmin(axis=0), np.arange(len(date))]
                frame[f"DaysToNearest{name}"] = np.abs(nearest_distance).astype("int16")
                frame[f"WeeksToNearest{name}"] = (np.abs(nearest_distance) / 7.0).astype("float32")

        return frame


class MarkdownFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        markdown_cols: tuple[str, ...] = MARKDOWN_COLS,
        add_total_markdown: bool = True,
        add_has_markdown: bool = True,
        add_log_markdowns: bool = True,
        add_holiday_interaction: bool = True,
        holiday_col: str = "IsHoliday",
    ):
        self.markdown_cols = markdown_cols
        self.add_total_markdown = add_total_markdown
        self.add_has_markdown = add_has_markdown
        self.add_log_markdowns = add_log_markdowns
        self.add_holiday_interaction = add_holiday_interaction
        self.holiday_col = holiday_col

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        markdown_cols = _existing_columns(frame, self.markdown_cols)

        if self.add_total_markdown and markdown_cols:
            frame["TotalMarkDown"] = frame[markdown_cols].sum(axis=1)

        if self.add_has_markdown:
            for col in markdown_cols:
                frame[f"Has{col}"] = (frame[col] > 0).astype("int8")
            if "TotalMarkDown" in frame.columns:
                frame["HasAnyMarkDown"] = (frame["TotalMarkDown"] > 0).astype("int8")

        if self.add_log_markdowns:
            for col in markdown_cols:
                frame[f"{col}_log1p"] = np.log1p(frame[col].clip(lower=0))
            if "TotalMarkDown" in frame.columns:
                frame["TotalMarkDown_log1p"] = np.log1p(frame["TotalMarkDown"].clip(lower=0))

        if self.add_holiday_interaction and self.holiday_col in frame.columns:
            if "TotalMarkDown" in frame.columns:
                frame["Holiday_TotalMarkDown"] = frame[self.holiday_col] * frame["TotalMarkDown"]
            for col in markdown_cols:
                frame[f"Holiday_{col}"] = frame[self.holiday_col] * frame[col]

        return frame


class InteractionFeatureTransformer(BaseEstimator, TransformerMixin):


    def __init__(
        self,
        interactions: tuple[tuple[str, ...], ...] = (("Store", "Dept"), ("Type", "Dept")),
        separator: str = "_",
        as_category: bool = True,
    ):
        self.interactions = interactions
        self.separator = separator
        self.as_category = as_category

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()

        for cols in self.interactions:
            if all(col in frame.columns for col in cols):
                new_col = self.separator.join(cols)
                values = frame[list(cols)].astype(str).agg(self.separator.join, axis=1)
                frame[new_col] = values.astype("category") if self.as_category else values

        return frame


class LagRollingFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        group_cols: tuple[str, ...] = ("Store", "Dept"),
        date_col: str = "Date",
        target_col: str = "Weekly_Sales",
        lags: tuple[int, ...] = (1, 4, 13, 52),
        rolling_windows: tuple[int, ...] = (4, 13),
        rolling_stats: tuple[str, ...] = ("mean", "std"),
        min_periods: int = 1,
    ):
        self.group_cols = group_cols
        self.date_col = date_col
        self.target_col = target_col
        self.lags = lags
        self.rolling_windows = rolling_windows
        self.rolling_stats = rolling_stats
        self.min_periods = min_periods

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.target_col not in X.columns:
            raise ValueError(
                f"{self.target_col!r} is required for lag/rolling features. "
                "For test data, append historical sales first or use recursive inference."
            )

        frame = X.copy().sort_values(list(self.group_cols) + [self.date_col])
        grouped = frame.groupby(list(self.group_cols), observed=True)[self.target_col]

        for lag in self.lags:
            frame[f"lag_{lag}"] = grouped.shift(lag)

        for window in self.rolling_windows:
            shifted = grouped.shift(1)
            rolling = shifted.groupby([frame[col] for col in self.group_cols], observed=True).rolling(
                window=window,
                min_periods=self.min_periods,
            )
            if "mean" in self.rolling_stats:
                frame[f"rolling_mean_{window}"] = rolling.mean().reset_index(level=list(range(len(self.group_cols))), drop=True)
            if "std" in self.rolling_stats:
                frame[f"rolling_std_{window}"] = rolling.std().reset_index(level=list(range(len(self.group_cols))), drop=True)
            if "min" in self.rolling_stats:
                frame[f"rolling_min_{window}"] = rolling.min().reset_index(level=list(range(len(self.group_cols))), drop=True)
            if "max" in self.rolling_stats:
                frame[f"rolling_max_{window}"] = rolling.max().reset_index(level=list(range(len(self.group_cols))), drop=True)

        return frame.sort_index()


class HistoricalAggregateTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        groupings: tuple[tuple[str, ...], ...] = (("Store",), ("Dept",), ("Store", "Dept"), ("Type", "Dept")),
        target_col: str = "Weekly_Sales",
        stats: tuple[str, ...] = ("mean", "median", "std"),
        fill_missing_with_global: bool = True,
    ):
        self.groupings = groupings
        self.target_col = target_col
        self.stats = stats
        self.fill_missing_with_global = fill_missing_with_global

    def fit(self, X: pd.DataFrame, y=None):
        if self.target_col not in X.columns:
            raise ValueError(f"{self.target_col!r} must be present when fitting aggregates.")

        self.global_stats_ = X[self.target_col].agg(list(self.stats)).to_dict()
        self.aggregate_frames_ = []

        for grouping in self.groupings:
            existing_grouping = tuple(col for col in grouping if col in X.columns)
            if not existing_grouping:
                continue
            prefix = "_".join(existing_grouping)
            agg = (
                X.groupby(list(existing_grouping), observed=True)[self.target_col]
                .agg(list(self.stats))
                .reset_index()
            )
            rename = {stat: f"{prefix}_{self.target_col}_{stat}" for stat in self.stats}
            agg = agg.rename(columns=rename)
            self.aggregate_frames_.append((existing_grouping, agg, rename))

        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()

        for grouping, agg, rename in self.aggregate_frames_:
            frame = frame.merge(agg, on=list(grouping), how="left", validate="many_to_one")
            if self.fill_missing_with_global:
                for stat, col in rename.items():
                    frame[col] = frame[col].fillna(self.global_stats_[stat])

        return frame


class ColumnDropper(BaseEstimator, TransformerMixin):

    def __init__(self, columns: tuple[str, ...] = ("Date", "Weekly_Sales"), errors: str = "ignore"):
        self.columns = columns
        self.errors = errors

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X.drop(columns=list(self.columns), errors=self.errors)


class FeatureImportanceSelector(BaseEstimator, TransformerMixin):

    def __init__(self, estimator, threshold: float = 0.0, fit_params: dict | None = None):
        self.estimator = estimator
        self.threshold = threshold
        self.fit_params = fit_params

    def fit(self, X: pd.DataFrame, y):
        fit_params = self.fit_params or {}
        self.estimator.fit(X, y, **fit_params)
        importances = getattr(self.estimator, "feature_importances_", None)
        if importances is None:
            raise ValueError("estimator must expose feature_importances_ after fit.")

        self.feature_importances_ = pd.Series(importances, index=X.columns).sort_values(ascending=False)
        self.selected_features_ = self.feature_importances_[
            self.feature_importances_ > self.threshold
        ].index.tolist()
        if not self.selected_features_:
            raise ValueError("No features passed the importance threshold.")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X[self.selected_features_].copy()


def make_walmart_lgbm_feature_pipeline(
    include_lag_features: bool = True,
    drop_target_and_date: bool = True,
) -> Pipeline:

    pre_processing = [
        ("clean", WalmartFeatureCleaner()),
        ("calendar", CalendarFeatureTransformer()),
        ("holiday", WalmartHolidayFeatureTransformer()),
        ("markdown", MarkdownFeatureTransformer()),
        ("interactions", InteractionFeatureTransformer()),
        ("aggregates", HistoricalAggregateTransformer()),
    ]

    if include_lag_features:
        pre_processing.append(("lags_rollings", LagRollingFeatureTransformer()))

    if drop_target_and_date:
        pre_processing.append(("drop_columns", ColumnDropper()))

    return Pipeline(pre_processing)


In [ ]:
feature_pipeline = make_walmart_lgbm_feature_pipeline(
    include_lag_features=True,
    drop_target_and_date=True
)

X_train_transformed = feature_pipeline.fit_transform(X_train, y_train)

X_val_transformed = feature_pipeline.transform(X_val)

sample_weights_train = np.where(is_holiday_train, 5, 1)
sample_weights_val = np.where(is_holiday_val, 5, 1)

hyperparameter_sets = [
    {'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': -1},
    {'n_estimators': 1000, 'learning_rate': 0.01, 'num_leaves': 63, 'max_depth': 10},
    {'n_estimators': 750, 'learning_rate': 0.03, 'num_leaves': 40, 'max_depth': 7}
]

results = []
best_model = None
best_mae = float('inf')

for i, params in enumerate(hyperparameter_sets):
    print(f"\nTraining LightGBM model with hyperparameter set {i+1}: {params}")
    lgbm = lgb.LGBMRegressor(objective='mae', random_state=42, **params)

    lgbm.fit(X_train_transformed, y_train, sample_weight=sample_weights_train)

    y_pred_val = lgbm.predict(X_val_transformed)
    weighted_mae = np.sum(np.abs(y_val - y_pred_val) * sample_weights_val) / np.sum(sample_weights_val)
    print(f"Validation Weighted MAE: {weighted_mae:.4f}")

    results.append({'params': params, 'weighted_mae': weighted_mae, 'model': lgbm})

    if weighted_mae < best_mae:
        best_mae = weighted_mae
        best_model = lgbm

print("\n--- Hyperparameter Tuning Results ---")
for res in results:
    print(f"Params: {res['params']}, Weighted MAE: {res['weighted_mae']:.4f}")

print(f"\nBest model achieved Weighted MAE: {best_mae:.4f} with parameters: {best_model.get_params()}")
print("LightGBM hyperparameter tuning complete.")

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(



Training LightGBM model with hyperparameter set 1: {'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': -1}
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.293918 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13151
[LightGBM] [Info] Number of data points in the train set: 337256, number of used features: 82
[LightGBM] [Info] Start training from score 7687.269531
Validation Weighted MAE: 2231.1697

Training LightGBM model with hyperparameter set 2: {'n_estimators': 1000, 'learning_rate': 0.01, 'num_leaves': 63, 'max_depth': 10}
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning